# Bond Liquidity Prediction: Complete ML Pipeline

This notebook recreates the entire ML pipeline from `comprehensive_dataset_final.parquet`.

**Includes:**
- Data loading from Google Drive
- Preprocessing and feature engineering
- Train/Validation/Test splits
- Elastic Net (with parallelized hyperparameter tuning)
- LightGBM with Optuna optimization

**Target Variable:** `bid_ask_spread_20d`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install Dependencies

In [ ]:
!pip install optuna lightgbm --quiet

## 3. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import gc
import time
import warnings
from joblib import Parallel, delayed

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("="*80)
print("BOND LIQUIDITY PREDICTION PIPELINE")
print("="*80)

# Check available CPU cores
n_cores = os.cpu_count()
print(f"\nAvailable CPU cores: {n_cores}")

BOND LIQUIDITY PREDICTION PIPELINE

Available CPU cores: 8


In [ ]:
print("\n" + "="*80)
print("STEP 1: LOADING DATA")
print("="*80)

# ⚠️ UPDATE THIS PATH TO MATCH YOUR GOOGLE DRIVE LOCATION
DATA_PATH = '/content/drive/MyDrive/comprehensive_dataset_final.parquet'

print(f"\n1️⃣ Loading data from: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)

print(f"   Shape: {df.shape}")
print(f"   Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"   Columns: {len(df.columns)}")

# Display basic info
print(f"\n📊 Dataset Overview:")
print(f"   Date range: {df['date'].min()} to {df['date'].max()}")
print(f"   Unique bonds (CUSIPs): {df['cusip'].nunique():,}")

# Check target variable
target_col = 'bid_ask_spread_20d'
print(f"\n🎯 Target Variable ({target_col}):")
print(f"   Non-null: {df[target_col].notna().sum():,} ({df[target_col].notna().mean()*100:.1f}%)")
print(f"   Mean: {df[target_col].mean():.6f}")
print(f"   Median: {df[target_col].median():.6f}")


STEP 1: LOADING DATA

1️⃣ Loading data from: /content/drive/MyDrive/comprehensive_dataset_final.parquet
   Shape: (2347177, 241)
   Memory: 5.49 GB
   Columns: 241

📊 Dataset Overview:
   Date range: 2015-01-31 00:00:00 to 2024-08-31 00:00:00
   Unique bonds (CUSIPs): 78,421

🎯 Target Variable (bid_ask_spread_20d):
   Non-null: 1,556,420 (66.3%)
   Mean: 0.008764
   Median: 0.005572


## 5. Data Preprocessing

In [ ]:
print("\n" + "="*80)
print("STEP 2: DATA PREPROCESSING")
print("="*80)

# 2.1: Drop rows with missing target variable
print("\n2️⃣ Dropping rows with missing bid-ask spread...")
print(f"   Before: {len(df):,} rows")
df_clean = df.dropna(subset=[target_col]).copy()
print(f"   After:  {len(df_clean):,} rows")
print(f"   Dropped: {len(df) - len(df_clean):,} rows ({(len(df) - len(df_clean))/len(df)*100:.1f}%)")

del df
gc.collect()

# 2.2: Winsorize target variable
print("\n3️⃣ Winsorizing bid-ask spread (0.1% / 99.9%)...")
print(f"   Before: Mean={df_clean[target_col].mean():.6f}, Max={df_clean[target_col].max():.6f}")

lower_bound = df_clean[target_col].quantile(0.001)
upper_bound = df_clean[target_col].quantile(0.999)
df_clean[target_col] = df_clean[target_col].clip(lower_bound, upper_bound)

print(f"   After:  Mean={df_clean[target_col].mean():.6f}, Max={df_clean[target_col].max():.6f}")

# 2.3: Ensure date is datetime
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean = df_clean.sort_values('date')

print(f"\n   Final preprocessed shape: {df_clean.shape}")


STEP 2: DATA PREPROCESSING

2️⃣ Dropping rows with missing bid-ask spread...
   Before: 2,347,177 rows
   After:  1,556,420 rows
   Dropped: 790,757 rows (33.7%)

3️⃣ Winsorizing bid-ask spread (0.1% / 99.9%)...
   Before: Mean=0.008764, Max=28.728237
   After:  Mean=0.007779, Max=0.314261

   Final preprocessed shape: (1556420, 241)


In [ ]:
print("\n" + "="*80)
print("STEP 3: FEATURE IDENTIFICATION")
print("="*80)

# ID columns (not features)
id_cols = ['cusip', 'issuer_cusip', 'date']

# Other target variables (not features for spread prediction)
other_targets = ['ytm', 'ret_eom']

# Leakage features to exclude
LEAKAGE_FEATURES = [
    'ytm_spread_to_10y',
    'ytm_spread_to_ig',
    'ytm_spread_to_comparable',
    'ytm_rank',
    'coupon_to_ytm',
]

# Identify all feature columns
exclude_cols = id_cols + [target_col] + other_targets + LEAKAGE_FEATURES
all_features = [c for c in df_clean.columns if c not in exclude_cols]

# Get numeric features only
numeric_features = df_clean[all_features].select_dtypes(include=[np.number]).columns.tolist()

print(f"4️⃣ Feature Summary:")
print(f"   Total columns: {len(df_clean.columns)}")
print(f"   ID columns: {len(id_cols)}")
print(f"   Leakage features removed: {len(LEAKAGE_FEATURES)}")
print(f"   Final numeric features: {len(numeric_features)}")


STEP 3: FEATURE IDENTIFICATION
4️⃣ Feature Summary:
   Total columns: 241
   ID columns: 3
   Leakage features removed: 5
   Final numeric features: 216


In [ ]:
print("\n" + "="*80)
print("STEP 4: CREATING TRAIN/VALIDATION/TEST SPLITS")
print("="*80)

# Time-based splits following Gu, Kelly, Xiu (2020) methodology
TRAIN_END = '2021-12-31'
VAL_END = '2023-06-30'

print(f"\n5️⃣ Time-based splitting:")
print(f"   Training:   up to {TRAIN_END}")
print(f"   Validation: {TRAIN_END} to {VAL_END}")
print(f"   Test:       after {VAL_END}")

# Create splits
train_mask = df_clean['date'] <= TRAIN_END
val_mask = (df_clean['date'] > TRAIN_END) & (df_clean['date'] <= VAL_END)
test_mask = df_clean['date'] > VAL_END

df_train = df_clean[train_mask].copy()
df_val = df_clean[val_mask].copy()
df_test = df_clean[test_mask].copy()

print(f"\n   Split sizes:")
print(f"      Train: {len(df_train):,} ({len(df_train)/len(df_clean)*100:.1f}%)")
print(f"      Val:   {len(df_val):,} ({len(df_val)/len(df_clean)*100:.1f}%)")
print(f"      Test:  {len(df_test):,} ({len(df_test)/len(df_clean)*100:.1f}%)")


STEP 4: CREATING TRAIN/VALIDATION/TEST SPLITS

5️⃣ Time-based splitting:
   Training:   up to 2021-12-31
   Validation: 2021-12-31 to 2023-06-30
   Test:       after 2023-06-30

   Split sizes:
      Train: 973,435 (62.5%)
      Val:   386,843 (24.9%)
      Test:  196,142 (12.6%)


## 8. Save Parquet Files

In [ ]:
print("\n" + "="*80)
print("STEP 5: SAVING PARQUET FILES")
print("="*80)

print("\n6️⃣ Saving tree model parquet files...")

# For tree models: keep all features, minimal preprocessing
df_train.to_parquet('tree_ytm_train.parquet', index=False)
df_val.to_parquet('tree_ytm_val.parquet', index=False)
df_test.to_parquet('tree_ytm_test.parquet', index=False)

print("   ✅ Saved: tree_ytm_train.parquet")
print("   ✅ Saved: tree_ytm_val.parquet")
print("   ✅ Saved: tree_ytm_test.parquet")

print("\n7️⃣ Preparing Elastic Net data...")

# For Elastic Net: need numeric features only
enet_cols = id_cols + [target_col] + other_targets + numeric_features

df_train_enet = df_train[enet_cols].copy()
df_val_enet = df_val[enet_cols].copy()
df_test_enet = df_test[enet_cols].copy()

# Save elastic net parquet files
df_train_enet.to_parquet('enet_ytm_train.parquet', index=False)
df_val_enet.to_parquet('enet_ytm_val.parquet', index=False)
df_test_enet.to_parquet('enet_ytm_test.parquet', index=False)

print("   ✅ Saved: enet_ytm_train.parquet")
print("   ✅ Saved: enet_ytm_val.parquet")
print("   ✅ Saved: enet_ytm_test.parquet")

# Clean up
del df_clean, df_train, df_val, df_test
del df_train_enet, df_val_enet, df_test_enet
gc.collect()

print("\n✅ Data preparation complete!")


STEP 5: SAVING PARQUET FILES

6️⃣ Saving tree model parquet files...
   ✅ Saved: tree_ytm_train.parquet
   ✅ Saved: tree_ytm_val.parquet
   ✅ Saved: tree_ytm_test.parquet

7️⃣ Preparing Elastic Net data...
   ✅ Saved: enet_ytm_train.parquet
   ✅ Saved: enet_ytm_val.parquet
   ✅ Saved: enet_ytm_test.parquet

✅ Data preparation complete!


## 10. LightGBM Model

In [ ]:
print("\n" + "="*80)
print("STEP 7: LIGHTGBM FOR BID-ASK SPREAD")
print("="*80)

# Reload tree data
print("\n1️⃣2️⃣ Loading LightGBM data...")

train_df = pd.read_parquet('tree_ytm_train.parquet')
val_df = pd.read_parquet('tree_ytm_val.parquet')
test_df = pd.read_parquet('tree_ytm_test.parquet')

print(f"   Train shape: {train_df.shape}")
print(f"   Val shape:   {val_df.shape}")
print(f"   Test shape:  {test_df.shape}")

# Prepare features for LightGBM
all_features_lgb = [c for c in train_df.columns if c not in id_cols + [target_col] + other_targets]
clean_features_lgb = [f for f in all_features_lgb if f not in LEAKAGE_FEATURES]

# Get only numeric features
numeric_features_lgb = train_df[clean_features_lgb].select_dtypes(include=[np.number]).columns.tolist()

print(f"   Features for LightGBM: {len(numeric_features_lgb)}")

# Prepare data
X_train_lgb = train_df[numeric_features_lgb]
X_val_lgb = val_df[numeric_features_lgb]
X_test_lgb = test_df[numeric_features_lgb]

y_train_lgb = train_df[target_col].values
y_val_lgb = val_df[target_col].values
y_test_lgb = test_df[target_col].values

# Create LightGBM datasets
lgb_train = lgb.Dataset(X_train_lgb, y_train_lgb)
lgb_val = lgb.Dataset(X_val_lgb, y_val_lgb, reference=lgb_train)

# Free memory
del train_df, val_df, test_df
gc.collect()


STEP 7: LIGHTGBM FOR BID-ASK SPREAD

1️⃣2️⃣ Loading LightGBM data...
   Train shape: (973435, 241)
   Val shape:   (386843, 241)
   Test shape:  (196142, 241)
   Features for LightGBM: 216


0

In [ ]:
# Optuna Hyperparameter Tuning
print("\n1️⃣3️⃣ Tuning LightGBM with Optuna (30 trials)...")

# Determine parallelization strategy
N_PARALLEL_TRIALS = min(2, n_cores)
THREADS_PER_MODEL = max(1, n_cores // N_PARALLEL_TRIALS)

print(f"   Running {N_PARALLEL_TRIALS} parallel trials with {THREADS_PER_MODEL} threads each")

def objective_spread(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'random_state': 42,
        'n_jobs': THREADS_PER_MODEL,
        'feature_pre_filter': False,

        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-6, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
    }

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=300,
        valid_sets=[lgb_val],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )

    y_pred = model.predict(X_val_lgb, num_iteration=model.best_iteration)
    return np.sqrt(mean_squared_error(y_val_lgb, y_pred))

# Progress callback
trial_count = [0]
def callback(study, trial):
    trial_count[0] += 1
    if trial_count[0] % 5 == 0:
        print(f"   Trial {trial_count[0]}/30, Best RMSE: {study.best_value:.6f}")

# Run optimization
study = optuna.create_study(direction='minimize', study_name='spread_lgb')
study.optimize(
    objective_spread,
    n_trials=30,
    n_jobs=N_PARALLEL_TRIALS,
    callbacks=[callback]
)

print(f"\n   ✅ Best LightGBM RMSE: {study.best_value:.6f}")
print(f"   Best parameters:")
for key, value in study.best_params.items():
    if isinstance(value, float):
        print(f"      {key}: {value:.6f}")
    else:
        print(f"      {key}: {value}")


1️⃣3️⃣ Tuning LightGBM with Optuna (30 trials)...
   Running 2 parallel trials with 4 threads each
   Trial 5/30, Best RMSE: 0.011141
   Trial 10/30, Best RMSE: 0.011141
   Trial 15/30, Best RMSE: 0.011107
   Trial 20/30, Best RMSE: 0.011012
   Trial 25/30, Best RMSE: 0.011012
   Trial 30/30, Best RMSE: 0.011008

   ✅ Best LightGBM RMSE: 0.011008
   Best parameters:
      num_leaves: 69
      learning_rate: 0.033106
      feature_fraction: 0.872384
      bagging_fraction: 0.831701
      bagging_freq: 4
      min_child_samples: 29
      reg_alpha: 0.633989
      reg_lambda: 0.000040
      max_depth: 9


In [ ]:
# Train Final LightGBM Model
print("\n1️⃣4️⃣ Training final LightGBM model...")

best_lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'random_state': 42,
    'n_jobs': -1,
    'feature_pre_filter': False,
    **study.best_params
}

final_lgb_model = lgb.train(
    best_lgb_params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_val],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

print(f"   Best iteration: {final_lgb_model.best_iteration}")

# Predictions
y_train_lgb_pred = final_lgb_model.predict(X_train_lgb, num_iteration=final_lgb_model.best_iteration)
y_val_lgb_pred = final_lgb_model.predict(X_val_lgb, num_iteration=final_lgb_model.best_iteration)
y_test_lgb_pred = final_lgb_model.predict(X_test_lgb, num_iteration=final_lgb_model.best_iteration)

# Metrics
lgb_train_rmse = np.sqrt(mean_squared_error(y_train_lgb, y_train_lgb_pred))
lgb_val_rmse = np.sqrt(mean_squared_error(y_val_lgb, y_val_lgb_pred))
lgb_test_rmse = np.sqrt(mean_squared_error(y_test_lgb, y_test_lgb_pred))
lgb_train_r2 = r2_score(y_train_lgb, y_train_lgb_pred)
lgb_val_r2 = r2_score(y_val_lgb, y_val_lgb_pred)
lgb_test_r2 = r2_score(y_test_lgb, y_test_lgb_pred)

print(f"\n   LightGBM Performance:")
print(f"      Train RMSE: {lgb_train_rmse:.6f}, R²: {lgb_train_r2:.4f}")
print(f"      Val RMSE:   {lgb_val_rmse:.6f}, R²: {lgb_val_r2:.4f}")
print(f"      Test RMSE:  {lgb_test_rmse:.6f}, R²: {lgb_test_r2:.4f}")

# Feature importance
importance_df = pd.DataFrame({
    'feature': numeric_features_lgb,
    'importance': final_lgb_model.feature_importance(importance_type='gain')
})
importance_df = importance_df.sort_values('importance', ascending=False)

print(f"\n   Top 15 features by importance:")
for _, row in importance_df.head(15).iterrows():
    print(f"      {row['feature']:40s}: {row['importance']:.2f}")


1️⃣4️⃣ Training final LightGBM model...
   Best iteration: 499

   LightGBM Performance:
      Train RMSE: 0.009420, R²: 0.5478
      Val RMSE:   0.010882, R²: 0.1973
      Test RMSE:  0.010796, R²: 0.1243

   Top 15 features by importance:
      log_issue_size                          : 224.26
      bond_age                                : 141.74
      time_to_maturity                        : 132.05
      days_since_trade                        : 97.11
      avg_spread_monthly                      : 60.24
      price_ma_12m                            : 59.16
      price_eom                               : 41.11
      price_eom_lag1m                         : 31.92
      offering_amt                            : 24.80
      price_ma_6m                             : 24.70
      vix_rating_interaction                  : 23.82
      avg_spread_3m                           : 23.62
      avg_spread_monthly_lag1m                : 19.67
      price_eom_lag6m                         : 19.26

## 11. Final Summary

In [47]:
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)

print(f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                         MODEL COMPARISON                                      ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Model        │  Train RMSE  │  Val RMSE   │  Test RMSE  │  Test R²          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  LightGBM     │  {lgb_train_rmse:10.6f}  │  {lgb_val_rmse:10.6f}  │  {lgb_test_rmse:10.6f}  │  {lgb_test_r2:8.4f}         ║
╚══════════════════════════════════════════════════════════════════════════════╝


📊 LightGBM Configuration:
   Best iteration: {final_lgb_model.best_iteration}
   Num leaves: {study.best_params.get('num_leaves', 'N/A')}
   Learning rate: {study.best_params.get('learning_rate', 0):.4f}
   Max depth: {study.best_params.get('max_depth', 'N/A')}

📁 Files Created:
   • tree_ytm_train.parquet
   • tree_ytm_val.parquet
   • tree_ytm_test.parquet
   • enet_ytm_train.parquet
   • enet_ytm_val.parquet
   • enet_ytm_test.parquet
""")

print("="*80)
print("✅ PIPELINE COMPLETE!")
print("="*80)


FINAL RESULTS SUMMARY

╔══════════════════════════════════════════════════════════════════════════════╗
║                         MODEL COMPARISON                                      ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Model        │  Train RMSE  │  Val RMSE   │  Test RMSE  │  Test R²          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  LightGBM     │    0.009420  │    0.010882  │    0.010796  │    0.1243         ║
╚══════════════════════════════════════════════════════════════════════════════╝


📊 LightGBM Configuration:
   Best iteration: 499
   Num leaves: 69
   Learning rate: 0.0331
   Max depth: 9

📁 Files Created:
   • tree_ytm_train.parquet
   • tree_ytm_val.parquet
   • tree_ytm_test.parquet
   • enet_ytm_train.parquet
   • enet_ytm_val.parquet
   • enet_ytm_test.parquet

✅ PIPELINE COMPLETE!


In [ ]:
"""
================================================================================
MULTINOMIAL LOGISTIC REGRESSION FOR BOND LIQUIDITY CLASSIFICATION
================================================================================

This module implements direct classification of bonds into liquidity categories
(Liquid, Neutral, Non-liquid) using multinomial logistic regression.

Key Considerations:
    1. The target variable (bid-ask spread) is fundamentally continuous and ordinal
    2. Multinomial logistic regression treats categories as unordered (nominal)
    3. For comparison purposes, we also implement ordinal logistic regression

Academic Note:
    The regression + discretization approach (your Approach 1) is generally
    preferred in empirical asset pricing because it preserves information
    about spread magnitudes. Direct classification is included here for
    completeness and comparison.

Author: Jason's ML for Finance Course
================================================================================
"""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, log_loss, roc_auc_score
)
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def create_liquidity_categories(spread_values, method='tercile', custom_thresholds=None):
    """
    Convert continuous bid-ask spreads to categorical liquidity labels.

    Parameters
    ----------
    spread_values : array-like
        Continuous bid-ask spread values
    method : str
        'tercile' : Equal-frequency bins (33.3% each)
        'custom'  : Use provided custom_thresholds
    custom_thresholds : tuple, optional
        (low_threshold, high_threshold) for custom binning

    Returns
    -------
    categories : np.array
        0 = Liquid, 1 = Neutral, 2 = Non-liquid
    thresholds : tuple
        (low_threshold, high_threshold) used for binning
    """
    spread_array = np.array(spread_values)
    valid_spreads = spread_array[~np.isnan(spread_array)]

    if method == 'tercile':
        low_thresh = np.percentile(valid_spreads, 33.33)
        high_thresh = np.percentile(valid_spreads, 66.67)
    elif method == 'custom' and custom_thresholds is not None:
        low_thresh, high_thresh = custom_thresholds
    else:
        raise ValueError(f"Unknown method: {method}")

    categories = np.zeros(len(spread_array), dtype=int)
    categories[spread_array <= low_thresh] = 0       # Liquid
    categories[(spread_array > low_thresh) &
               (spread_array <= high_thresh)] = 1    # Neutral
    categories[spread_array > high_thresh] = 2       # Non-liquid

    return categories, (low_thresh, high_thresh)


def print_classification_metrics(y_true, y_pred, y_pred_proba=None,
                                  thresholds=None, dataset_name='Test'):
    """
    Print comprehensive classification metrics.

    Parameters
    ----------
    y_true : array
        True categorical labels (0, 1, 2)
    y_pred : array
        Predicted categorical labels
    y_pred_proba : array, optional
        Predicted probabilities for each class (n_samples x 3)
    thresholds : tuple, optional
        (low, high) thresholds for interpreting categories
    dataset_name : str
        Name of the dataset for display
    """
    class_names = ['Liquid', 'Neutral', 'Non-liquid']

    print(f"\n{'='*70}")
    print(f"CLASSIFICATION RESULTS: {dataset_name} Set")
    print(f"{'='*70}")

    if thresholds:
        print(f"\nThreshold Definitions:")
        print(f"   Liquid:      spread <= {thresholds[0]:.6f}")
        print(f"   Neutral:     {thresholds[0]:.6f} < spread <= {thresholds[1]:.6f}")
        print(f"   Non-liquid:  spread > {thresholds[1]:.6f}")

    # Overall Metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average='macro')
    recall_macro = recall_score(y_true, y_pred, average='macro')
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')

    print(f"\nOverall Performance:")
    print(f"   Accuracy:           {accuracy:.4f}")
    print(f"   Macro Precision:    {precision_macro:.4f}")
    print(f"   Macro Recall:       {recall_macro:.4f}")
    print(f"   Macro F1 Score:     {f1_macro:.4f}")
    print(f"   Weighted F1 Score:  {f1_weighted:.4f}")

    # Log Loss (if probabilities available)
    if y_pred_proba is not None:
        try:
            logloss = log_loss(y_true, y_pred_proba)
            print(f"   Log Loss:           {logloss:.4f}")
        except:
            pass

    # Per-Class Metrics
    print(f"\nPer-Class Performance:")
    report_dict = classification_report(y_true, y_pred,
                                        target_names=class_names,
                                        output_dict=True)

    for class_name in class_names:
        metrics = report_dict[class_name]
        print(f"   {class_name:12s}  Precision: {metrics['precision']:.4f}  "
              f"Recall: {metrics['recall']:.4f}  F1: {metrics['f1-score']:.4f}  "
              f"Support: {int(metrics['support']):,}")

    # Confusion Matrix
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)

    print("                    Predicted")
    print("                    Liquid   Neutral  Non-liq")
    row_labels = ['Liquid    ', 'Neutral   ', 'Non-liquid']
    for i, label in enumerate(row_labels):
        prefix = "Actual " if i == 0 else "       "
        print(f"{prefix}{label}  [{cm[i,0]:7,d}  {cm[i,1]:7,d}  {cm[i,2]:7,d}]")

    # Error Analysis: Adjacent vs. Severe Misclassifications
    print(f"\nError Analysis:")
    total_errors = np.sum(y_true != y_pred)
    adjacent_errors = cm[0,1] + cm[1,0] + cm[1,2] + cm[2,1]  # Off by 1 category
    severe_errors = cm[0,2] + cm[2,0]  # Liquid<->Non-liquid confusion

    print(f"   Total Misclassifications:  {total_errors:,} ({total_errors/len(y_true)*100:.1f}%)")
    print(f"   Adjacent Errors (off-by-1): {adjacent_errors:,} ({adjacent_errors/len(y_true)*100:.1f}%)")
    print(f"   Severe Errors (extreme):    {severe_errors:,} ({severe_errors/len(y_true)*100:.1f}%)")

    return {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'confusion_matrix': cm,
        'adjacent_errors': adjacent_errors,
        'severe_errors': severe_errors
    }


# ============================================================================
# APPROACH 2: MULTINOMIAL LOGISTIC REGRESSION
# ============================================================================

def train_multinomial_logistic_regression(
    X_train, y_train_continuous,
    X_val, y_val_continuous,
    X_test, y_test_continuous,
    method='tercile',
    use_training_thresholds=True,
    regularization_values=None,
    verbose=True
):
    """
    Train multinomial logistic regression for liquidity classification.

    This function:
    1. Converts continuous spreads to categories using TRAINING data thresholds
    2. Tunes regularization strength (C parameter) on validation set
    3. Trains final model and evaluates on all splits

    Parameters
    ----------
    X_train, X_val, X_test : np.array
        Feature matrices (should be standardized)
    y_train_continuous, y_val_continuous, y_test_continuous : np.array
        Continuous bid-ask spread values
    method : str
        Categorization method ('tercile')
    use_training_thresholds : bool
        If True, use thresholds from training data for val/test (prevents leakage)
    regularization_values : list, optional
        C values to try (inverse regularization strength)
    verbose : bool
        Print detailed output

    Returns
    -------
    results : dict
        Contains trained model, predictions, probabilities, and all metrics
    """

    if verbose:
        print("\n" + "="*70)
        print("MULTINOMIAL LOGISTIC REGRESSION FOR LIQUIDITY CLASSIFICATION")
        print("="*70)
        print("\nNote: This treats categories as UNORDERED (nominal).")
        print("      Liquid, Neutral, Non-liquid ordering is not enforced.")

    # -------------------------------------------------------------------------
    # Step 1: Create Categorical Labels
    # -------------------------------------------------------------------------
    if verbose:
        print("\n" + "-"*50)
        print("STEP 1: Creating Liquidity Categories")
        print("-"*50)

    # CRITICAL: Compute thresholds from TRAINING data only (avoid data leakage)
    y_train_cat, train_thresholds = create_liquidity_categories(
        y_train_continuous, method=method
    )

    if use_training_thresholds:
        # Apply training thresholds to validation and test
        y_val_cat, _ = create_liquidity_categories(
            y_val_continuous, method='custom', custom_thresholds=train_thresholds
        )
        y_test_cat, _ = create_liquidity_categories(
            y_test_continuous, method='custom', custom_thresholds=train_thresholds
        )
        thresholds_used = train_thresholds
        if verbose:
            print("\n   Using TRAINING thresholds for all splits (no data leakage)")
    else:
        # Use each split's own thresholds (for comparison only)
        y_val_cat, val_thresholds = create_liquidity_categories(
            y_val_continuous, method=method
        )
        y_test_cat, test_thresholds = create_liquidity_categories(
            y_test_continuous, method=method
        )
        thresholds_used = train_thresholds
        if verbose:
            print("\n   WARNING: Using per-split thresholds (potential data leakage)")

    if verbose:
        print(f"\n   Thresholds (from training data):")
        print(f"      Liquid:     spread <= {train_thresholds[0]:.6f}")
        print(f"      Non-liquid: spread >  {train_thresholds[1]:.6f}")

        print(f"\n   Class Distribution:")
        for name, y_cat in [('Training', y_train_cat),
                            ('Validation', y_val_cat),
                            ('Test', y_test_cat)]:
            unique, counts = np.unique(y_cat, return_counts=True)
            total = len(y_cat)
            class_dist = [f"{['Liquid','Neutral','Non-liq'][u]}={c:,} ({c/total*100:.1f}%)"
                         for u, c in zip(unique, counts)]
            print(f"      {name:10s}: {', '.join(class_dist)}")

    # -------------------------------------------------------------------------
    # Step 2: Hyperparameter Tuning (Regularization Strength)
    # -------------------------------------------------------------------------
    if verbose:
        print("\n" + "-"*50)
        print("STEP 2: Tuning Regularization (C parameter)")
        print("-"*50)
        print("\n   C = inverse regularization strength (higher C = less regularization)")

    if regularization_values is None:
        regularization_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

    best_f1 = 0
    best_C = None
    tuning_results = []

    for C in regularization_values:
        model = LogisticRegression(
            C=C,
            penalty='l2',
            multi_class='multinomial',
            solver='lbfgs',
            max_iter=2000,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train_cat)
        y_val_pred = model.predict(X_val)

        acc = accuracy_score(y_val_cat, y_val_pred)
        f1 = f1_score(y_val_cat, y_val_pred, average='macro')

        tuning_results.append({'C': C, 'accuracy': acc, 'f1_macro': f1})

        if verbose:
            marker = " <-- BEST" if f1 > best_f1 else ""
            print(f"      C = {C:8.4f}:  Accuracy = {acc:.4f},  Macro F1 = {f1:.4f}{marker}")

        if f1 > best_f1:
            best_f1 = f1
            best_C = C

    if verbose:
        print(f"\n   Selected C: {best_C}")

    # -------------------------------------------------------------------------
    # Step 3: Train Final Model
    # -------------------------------------------------------------------------
    if verbose:
        print("\n" + "-"*50)
        print("STEP 3: Training Final Model")
        print("-"*50)

    final_model = LogisticRegression(
        C=best_C,
        penalty='l2',
        multi_class='multinomial',
        solver='lbfgs',
        max_iter=2000,
        random_state=42,
        n_jobs=-1
    )

    final_model.fit(X_train, y_train_cat)

    if verbose:
        print(f"\n   Model converged in {final_model.n_iter_[0]} iterations")

    # -------------------------------------------------------------------------
    # Step 4: Generate Predictions and Probabilities
    # -------------------------------------------------------------------------
    y_train_pred = final_model.predict(X_train)
    y_val_pred = final_model.predict(X_val)
    y_test_pred = final_model.predict(X_test)

    y_train_proba = final_model.predict_proba(X_train)
    y_val_proba = final_model.predict_proba(X_val)
    y_test_proba = final_model.predict_proba(X_test)

    # -------------------------------------------------------------------------
    # Step 5: Evaluate on All Splits
    # -------------------------------------------------------------------------
    if verbose:
        print("\n" + "-"*50)
        print("STEP 4: Evaluation Results")
        print("-"*50)

    metrics = {}

    for split_name, y_true, y_pred, y_proba in [
        ('Training', y_train_cat, y_train_pred, y_train_proba),
        ('Validation', y_val_cat, y_val_pred, y_val_proba),
        ('Test', y_test_cat, y_test_pred, y_test_proba)
    ]:
        if verbose:
            metrics[split_name] = print_classification_metrics(
                y_true, y_pred, y_proba, thresholds_used, split_name
            )
        else:
            metrics[split_name] = {
                'accuracy': accuracy_score(y_true, y_pred),
                'f1_macro': f1_score(y_true, y_pred, average='macro'),
                'confusion_matrix': confusion_matrix(y_true, y_pred)
            }

    # -------------------------------------------------------------------------
    # Step 6: Coefficient Analysis
    # -------------------------------------------------------------------------
    if verbose:
        print("\n" + "-"*50)
        print("STEP 5: Coefficient Analysis")
        print("-"*50)

        # Get coefficients: shape is (n_classes, n_features)
        coefs = final_model.coef_

        print("\n   Feature importance by absolute coefficient sum across classes:")
        print("   (Higher = more influential for classification)")

        # Sum absolute coefficients across classes
        importance = np.sum(np.abs(coefs), axis=0)

        # If feature names available (would need to pass them)
        print(f"\n   Top 10 features by importance:")
        top_indices = np.argsort(importance)[::-1][:10]
        for rank, idx in enumerate(top_indices, 1):
            print(f"      {rank:2d}. Feature {idx:3d}: {importance[idx]:.4f}")

    # -------------------------------------------------------------------------
    # Compile Results
    # -------------------------------------------------------------------------
    results = {
        'model': final_model,
        'best_C': best_C,
        'thresholds': thresholds_used,
        'predictions': {
            'train': y_train_pred,
            'val': y_val_pred,
            'test': y_test_pred
        },
        'probabilities': {
            'train': y_train_proba,
            'val': y_val_proba,
            'test': y_test_proba
        },
        'true_labels': {
            'train': y_train_cat,
            'val': y_val_cat,
            'test': y_test_cat
        },
        'metrics': metrics,
        'tuning_results': tuning_results,
        'coefficients': final_model.coef_,
        'intercept': final_model.intercept_
    }

    return results


# ============================================================================
# COMPARISON: REGRESSION + DISCRETIZATION VS. DIRECT CLASSIFICATION
# ============================================================================

def compare_approaches(
    y_test_continuous,           # True continuous spreads
    y_test_regression_pred,      # Predictions from regression model
    y_test_logistic_pred,        # Predictions from logistic regression
    y_test_logistic_proba,       # Probabilities from logistic regression
    method='tercile'
):
    """
    Compare Regression+Discretization vs Direct Classification approaches.

    Parameters
    ----------
    y_test_continuous : array
        True continuous bid-ask spreads
    y_test_regression_pred : array
        Continuous predictions from regression model (Elastic Net, LightGBM, etc.)
    y_test_logistic_pred : array
        Categorical predictions from multinomial logistic regression
    y_test_logistic_proba : array
        Class probabilities from logistic regression
    method : str
        Categorization method

    Returns
    -------
    comparison : dict
        Side-by-side metrics for both approaches
    """
    print("\n" + "="*70)
    print("COMPARISON: REGRESSION+DISCRETIZATION vs DIRECT CLASSIFICATION")
    print("="*70)

    # Create true categories
    y_test_cat_true, thresholds = create_liquidity_categories(
        y_test_continuous, method=method
    )

    # Discretize regression predictions
    y_test_regression_cat = np.zeros(len(y_test_regression_pred), dtype=int)
    y_test_regression_cat[y_test_regression_pred <= thresholds[0]] = 0
    y_test_regression_cat[(y_test_regression_pred > thresholds[0]) &
                          (y_test_regression_pred <= thresholds[1])] = 1
    y_test_regression_cat[y_test_regression_pred > thresholds[1]] = 2

    print(f"\nThresholds: Liquid <= {thresholds[0]:.6f}, Non-liquid > {thresholds[1]:.6f}")

    # Calculate metrics for both approaches
    print("\n" + "-"*50)
    print("APPROACH 1: Regression Model + Discretization")
    print("-"*50)

    reg_metrics = {
        'accuracy': accuracy_score(y_test_cat_true, y_test_regression_cat),
        'precision_macro': precision_score(y_test_cat_true, y_test_regression_cat, average='macro'),
        'recall_macro': recall_score(y_test_cat_true, y_test_regression_cat, average='macro'),
        'f1_macro': f1_score(y_test_cat_true, y_test_regression_cat, average='macro'),
        'confusion_matrix': confusion_matrix(y_test_cat_true, y_test_regression_cat)
    }

    print(f"   Accuracy:        {reg_metrics['accuracy']:.4f}")
    print(f"   Macro Precision: {reg_metrics['precision_macro']:.4f}")
    print(f"   Macro Recall:    {reg_metrics['recall_macro']:.4f}")
    print(f"   Macro F1:        {reg_metrics['f1_macro']:.4f}")

    print("\n" + "-"*50)
    print("APPROACH 2: Direct Multinomial Logistic Regression")
    print("-"*50)

    log_metrics = {
        'accuracy': accuracy_score(y_test_cat_true, y_test_logistic_pred),
        'precision_macro': precision_score(y_test_cat_true, y_test_logistic_pred, average='macro'),
        'recall_macro': recall_score(y_test_cat_true, y_test_logistic_pred, average='macro'),
        'f1_macro': f1_score(y_test_cat_true, y_test_logistic_pred, average='macro'),
        'log_loss': log_loss(y_test_cat_true, y_test_logistic_proba),
        'confusion_matrix': confusion_matrix(y_test_cat_true, y_test_logistic_pred)
    }

    print(f"   Accuracy:        {log_metrics['accuracy']:.4f}")
    print(f"   Macro Precision: {log_metrics['precision_macro']:.4f}")
    print(f"   Macro Recall:    {log_metrics['recall_macro']:.4f}")
    print(f"   Macro F1:        {log_metrics['f1_macro']:.4f}")
    print(f"   Log Loss:        {log_metrics['log_loss']:.4f}")

    # Summary Comparison
    print("\n" + "-"*50)
    print("SUMMARY COMPARISON")
    print("-"*50)

    print(f"\n{'Metric':<20} {'Regression+Disc':>15} {'Direct Logistic':>15} {'Difference':>12}")
    print("-"*62)

    for metric in ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']:
        reg_val = reg_metrics[metric]
        log_val = log_metrics[metric]
        diff = log_val - reg_val
        diff_str = f"{diff:+.4f}" if diff != 0 else "   ---"
        print(f"{metric:<20} {reg_val:>15.4f} {log_val:>15.4f} {diff_str:>12}")

    # Winner
    if reg_metrics['f1_macro'] > log_metrics['f1_macro']:
        winner = "Regression + Discretization"
        margin = reg_metrics['f1_macro'] - log_metrics['f1_macro']
    else:
        winner = "Direct Classification"
        margin = log_metrics['f1_macro'] - reg_metrics['f1_macro']

    print(f"\n   Winner (by F1): {winner} (+{margin:.4f})")

    return {
        'regression_discretization': reg_metrics,
        'direct_classification': log_metrics
    }


#==============================================================================
# FAST MULTINOMIAL LOGISTIC REGRESSION (SGD-based)
#==============================================================================

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
import time

print("\n" + "="*70)
print("MULTINOMIAL LOGISTIC REGRESSION (SGD - FAST)")
print("="*70)

# Create categories using TRAINING thresholds
y_train_cat, train_thresholds = create_liquidity_categories(y_train, method='tercile')

# Apply training thresholds to val/test (avoid leakage)
y_val_cat = np.zeros(len(y_val), dtype=int)
y_val_cat[y_val <= train_thresholds[0]] = 0
y_val_cat[(y_val > train_thresholds[0]) & (y_val <= train_thresholds[1])] = 1
y_val_cat[y_val > train_thresholds[1]] = 2

y_test_cat = np.zeros(len(y_test), dtype=int)
y_test_cat[y_test <= train_thresholds[0]] = 0
y_test_cat[(y_test > train_thresholds[0]) & (y_test <= train_thresholds[1])] = 1
y_test_cat[y_test > train_thresholds[1]] = 2

print(f"\nThresholds: Liquid <= {train_thresholds[0]:.6f}, Non-liquid > {train_thresholds[1]:.6f}")

# Tune regularization (alpha = 1/C, so smaller alpha = less regularization)
print("\nTuning alpha (regularization)...")
start = time.time()

best_f1, best_alpha = 0, 1e-4
for alpha in [1e-5, 1e-4, 1e-3, 1e-2, 0.1]:
    model = SGDClassifier(
        loss='log_loss',           # This makes it logistic regression
        penalty='l2',
        alpha=alpha,
        max_iter=1000,
        tol=1e-3,
        random_state=42,
        n_jobs=-1,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    model.fit(X_train, y_train_cat)
    f1 = f1_score(y_val_cat, model.predict(X_val), average='macro')
    marker = " <-- BEST" if f1 > best_f1 else ""
    print(f"   alpha={alpha:.0e}: Macro F1 = {f1:.4f}{marker}")
    if f1 > best_f1:
        best_f1, best_alpha = f1, alpha

print(f"\nBest alpha: {best_alpha:.0e}")
print(f"Tuning time: {time.time() - start:.1f} seconds")

# Train final model
print("\nTraining final model...")
start = time.time()

logistic_model = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=best_alpha,
    max_iter=2000,
    tol=1e-4,
    random_state=42,
    n_jobs=-1
)
logistic_model.fit(X_train, y_train_cat)

print(f"Training time: {time.time() - start:.1f} seconds")

# Predictions
y_test_log_pred = logistic_model.predict(X_test)
y_test_log_proba = logistic_model.predict_proba(X_test)

# Results
print("\n" + "="*50)
print("LOGISTIC REGRESSION TEST RESULTS")
print("="*50)
print(f"Accuracy:        {accuracy_score(y_test_cat, y_test_log_pred):.4f}")
print(f"Macro Precision: {precision_score(y_test_cat, y_test_log_pred, average='macro'):.4f}")
print(f"Macro Recall:    {recall_score(y_test_cat, y_test_log_pred, average='macro'):.4f}")
print(f"Macro F1:        {f1_score(y_test_cat, y_test_log_pred, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(y_test_cat, y_test_log_pred,
                            target_names=['Liquid', 'Neutral', 'Non-liquid']))

print("Confusion Matrix:")
cm = confusion_matrix(y_test_cat, y_test_log_pred)
print("                 Predicted")
print("                 Liquid  Neutral  Non-liq")
print(f"Actual Liquid     [{cm[0,0]:6d}  {cm[0,1]:6d}  {cm[0,2]:6d}]")
print(f"       Neutral    [{cm[1,0]:6d}  {cm[1,1]:6d}  {cm[1,2]:6d}]")
print(f"       Non-liquid [{cm[2,0]:6d}  {cm[2,1]:6d}  {cm[2,2]:6d}]")


MULTINOMIAL LOGISTIC REGRESSION (SGD - FAST)

Thresholds: Liquid <= 0.003375, Non-liquid > 0.008142

Tuning alpha (regularization)...
   alpha=1e-05: Macro F1 = 0.4620 <-- BEST
   alpha=1e-04: Macro F1 = 0.4984 <-- BEST
   alpha=1e-03: Macro F1 = 0.5353 <-- BEST
   alpha=1e-02: Macro F1 = 0.5368 <-- BEST
   alpha=1e-01: Macro F1 = 0.5228

Best alpha: 1e-02
Tuning time: 78.5 seconds

Training final model...
Training time: 12.2 seconds

LOGISTIC REGRESSION TEST RESULTS
Accuracy:        0.5328
Macro Precision: 0.5230
Macro Recall:    0.5403
Macro F1:        0.5133

Classification Report:
              precision    recall  f1-score   support

      Liquid       0.53      0.73      0.61     60268
     Neutral       0.48      0.25      0.33     67950
  Non-liquid       0.56      0.64      0.60     67924

    accuracy                           0.53    196142
   macro avg       0.52      0.54      0.51    196142
weighted avg       0.52      0.53      0.51    196142

Confusion Matrix:
        

In [ ]:
#==============================================================================
# PART 2: BINARY CLASSIFICATION (LIQUID vs NON-LIQUID)
#==============================================================================

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
import time

print("\n" + "="*70)
print("BINARY LOGISTIC REGRESSION: LIQUID vs NON-LIQUID")
print("="*70)

# ------------------------------------------------------------------------------
# OPTION A: Median Split (uses all data)
# ------------------------------------------------------------------------------
print("\n--- OPTION A: Median Split ---")

# Threshold at median (50th percentile) from TRAINING data
median_threshold = np.median(y_train)
print(f"Threshold (training median): {median_threshold:.6f}")

# Create binary labels: 0 = Liquid (below median), 1 = Non-liquid (above median)
y_train_binary = (y_train > median_threshold).astype(int)
y_val_binary = (y_val > median_threshold).astype(int)
y_test_binary = (y_test > median_threshold).astype(int)

print(f"\nClass distribution:")
print(f"   Train:  Liquid={np.sum(y_train_binary==0):,}, Non-liquid={np.sum(y_train_binary==1):,}")
print(f"   Val:    Liquid={np.sum(y_val_binary==0):,}, Non-liquid={np.sum(y_val_binary==1):,}")
print(f"   Test:   Liquid={np.sum(y_test_binary==0):,}, Non-liquid={np.sum(y_test_binary==1):,}")

# Tune regularization
print("\nTuning alpha...")
start = time.time()

best_f1, best_alpha = 0, 1e-4
for alpha in [1e-5, 1e-4, 1e-3, 1e-2, 0.1]:
    model = SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=alpha,
        max_iter=1000,
        tol=1e-3,
        random_state=42,
        n_jobs=-1,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    model.fit(X_train, y_train_binary)
    f1 = f1_score(y_val_binary, model.predict(X_val))
    marker = " <-- BEST" if f1 > best_f1 else ""
    print(f"   alpha={alpha:.0e}: F1 = {f1:.4f}{marker}")
    if f1 > best_f1:
        best_f1, best_alpha = f1, alpha

print(f"\nTuning time: {time.time() - start:.1f} seconds")

# Train final model
print("\nTraining final model...")
start = time.time()

binary_model = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=best_alpha,
    max_iter=2000,
    tol=1e-4,
    random_state=42,
    n_jobs=-1
)
binary_model.fit(X_train, y_train_binary)

print(f"Training time: {time.time() - start:.1f} seconds")

# Predictions
y_test_pred_binary = binary_model.predict(X_test)
y_test_proba_binary = binary_model.predict_proba(X_test)[:, 1]  # P(Non-liquid)

# Results
print("\n" + "="*50)
print("BINARY CLASSIFICATION RESULTS (Median Split)")
print("="*50)
print(f"Accuracy:    {accuracy_score(y_test_binary, y_test_pred_binary):.4f}")
print(f"Precision:   {precision_score(y_test_binary, y_test_pred_binary):.4f}")
print(f"Recall:      {recall_score(y_test_binary, y_test_pred_binary):.4f}")
print(f"F1 Score:    {f1_score(y_test_binary, y_test_pred_binary):.4f}")
print(f"AUC-ROC:     {roc_auc_score(y_test_binary, y_test_proba_binary):.4f}")

print("\nClassification Report:")
print(classification_report(y_test_binary, y_test_pred_binary,
                            target_names=['Liquid', 'Non-liquid']))

print("Confusion Matrix:")
cm = confusion_matrix(y_test_binary, y_test_pred_binary)
print("                 Predicted")
print("                 Liquid  Non-liquid")
print(f"Actual Liquid     [{cm[0,0]:6d}    {cm[0,1]:6d}]")
print(f"       Non-liquid [{cm[1,0]:6d}    {cm[1,1]:6d}]")


# ------------------------------------------------------------------------------
# OPTION B: Exclude Middle Tercile (cleaner separation)
# ------------------------------------------------------------------------------
print("\n" + "="*70)
print("--- OPTION B: Exclude Middle Tercile ---")
print("="*70)

# Use tercile thresholds from training
low_thresh = np.percentile(y_train, 33.33)
high_thresh = np.percentile(y_train, 66.67)

print(f"Excluding spreads between {low_thresh:.6f} and {high_thresh:.6f}")

# Create masks for extreme terciles only
train_mask = (y_train <= low_thresh) | (y_train > high_thresh)
val_mask = (y_val <= low_thresh) | (y_val > high_thresh)
test_mask = (y_test <= low_thresh) | (y_test > high_thresh)

# Filter data
X_train_extreme = X_train[train_mask]
X_val_extreme = X_val[val_mask]
X_test_extreme = X_test[test_mask]

# Binary labels: 0 = Liquid (bottom tercile), 1 = Non-liquid (top tercile)
y_train_extreme = (y_train[train_mask] > high_thresh).astype(int)
y_val_extreme = (y_val[val_mask] > high_thresh).astype(int)
y_test_extreme = (y_test[test_mask] > high_thresh).astype(int)

print(f"\nFiltered sample sizes:")
print(f"   Train: {len(y_train_extreme):,} (was {len(y_train):,})")
print(f"   Val:   {len(y_val_extreme):,} (was {len(y_val):,})")
print(f"   Test:  {len(y_test_extreme):,} (was {len(y_test):,})")

# Tune
print("\nTuning alpha...")
start = time.time()

best_f1, best_alpha = 0, 1e-4
for alpha in [1e-5, 1e-4, 1e-3, 1e-2, 0.1]:
    model = SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=alpha,
        max_iter=1000,
        tol=1e-3,
        random_state=42,
        n_jobs=-1,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    model.fit(X_train_extreme, y_train_extreme)
    f1 = f1_score(y_val_extreme, model.predict(X_val_extreme))
    marker = " <-- BEST" if f1 > best_f1 else ""
    print(f"   alpha={alpha:.0e}: F1 = {f1:.4f}{marker}")
    if f1 > best_f1:
        best_f1, best_alpha = f1, alpha

# Train final
binary_model_extreme = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=best_alpha,
    max_iter=2000,
    tol=1e-4,
    random_state=42,
    n_jobs=-1
)
binary_model_extreme.fit(X_train_extreme, y_train_extreme)

# Predictions
y_test_pred_extreme = binary_model_extreme.predict(X_test_extreme)
y_test_proba_extreme = binary_model_extreme.predict_proba(X_test_extreme)[:, 1]

# Results
print("\n" + "="*50)
print("BINARY CLASSIFICATION RESULTS (Extreme Terciles)")
print("="*50)
print(f"Accuracy:    {accuracy_score(y_test_extreme, y_test_pred_extreme):.4f}")
print(f"Precision:   {precision_score(y_test_extreme, y_test_pred_extreme):.4f}")
print(f"Recall:      {recall_score(y_test_extreme, y_test_pred_extreme):.4f}")
print(f"F1 Score:    {f1_score(y_test_extreme, y_test_pred_extreme):.4f}")
print(f"AUC-ROC:     {roc_auc_score(y_test_extreme, y_test_proba_extreme):.4f}")

print("\nClassification Report:")
print(classification_report(y_test_extreme, y_test_pred_extreme,
                            target_names=['Liquid', 'Non-liquid']))

print("Confusion Matrix:")
cm = confusion_matrix(y_test_extreme, y_test_pred_extreme)
print("                 Predicted")
print("                 Liquid  Non-liquid")
print(f"Actual Liquid     [{cm[0,0]:6d}    {cm[0,1]:6d}]")
print(f"       Non-liquid [{cm[1,0]:6d}    {cm[1,1]:6d}]")


# ------------------------------------------------------------------------------
# COMPARISON SUMMARY
# ------------------------------------------------------------------------------
print("\n" + "="*70)
print("SUMMARY: BINARY CLASSIFICATION APPROACHES")
print("="*70)

print(f"\n{'Approach':<30} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'AUC':>10}")
print("-"*80)

# Median split results
acc_med = accuracy_score(y_test_binary, y_test_pred_binary)
prec_med = precision_score(y_test_binary, y_test_pred_binary)
rec_med = recall_score(y_test_binary, y_test_pred_binary)
f1_med = f1_score(y_test_binary, y_test_pred_binary)
auc_med = roc_auc_score(y_test_binary, y_test_proba_binary)
print(f"{'Median Split (all data)':<30} {acc_med:>10.4f} {prec_med:>10.4f} {rec_med:>10.4f} {f1_med:>10.4f} {auc_med:>10.4f}")

# Extreme terciles results
acc_ext = accuracy_score(y_test_extreme, y_test_pred_extreme)
prec_ext = precision_score(y_test_extreme, y_test_pred_extreme)
rec_ext = recall_score(y_test_extreme, y_test_pred_extreme)
f1_ext = f1_score(y_test_extreme, y_test_pred_extreme)
auc_ext = roc_auc_score(y_test_extreme, y_test_proba_extreme)
print(f"{'Extreme Terciles (no middle)':<30} {acc_ext:>10.4f} {prec_ext:>10.4f} {rec_ext:>10.4f} {f1_ext:>10.4f} {auc_ext:>10.4f}")

print("\nNote: Extreme terciles should have HIGHER accuracy since we removed")
print("      the ambiguous middle category, but uses fewer test samples.")


BINARY LOGISTIC REGRESSION: LIQUID vs NON-LIQUID

--- OPTION A: Median Split ---
Threshold (training median): 0.005471

Class distribution:
   Train:  Liquid=486,722, Non-liquid=486,713
   Val:    Liquid=183,456, Non-liquid=203,387
   Test:   Liquid=96,935, Non-liquid=99,207

Tuning alpha...
   alpha=1e-05: F1 = 0.6218 <-- BEST
   alpha=1e-04: F1 = 0.6559 <-- BEST
   alpha=1e-03: F1 = 0.6657 <-- BEST
   alpha=1e-02: F1 = 0.7048 <-- BEST
   alpha=1e-01: F1 = 0.7027

Tuning time: 60.7 seconds

Training final model...
Training time: 9.9 seconds

BINARY CLASSIFICATION RESULTS (Median Split)
Accuracy:    0.7013
Precision:   0.7281
Recall:      0.6535
F1 Score:    0.6888
AUC-ROC:     0.7813

Classification Report:
              precision    recall  f1-score   support

      Liquid       0.68      0.75      0.71     96935
  Non-liquid       0.73      0.65      0.69     99207

    accuracy                           0.70    196142
   macro avg       0.70      0.70      0.70    196142
weighted 

In [ ]:
#==============================================================================
# FOUR-TIER LOGISTIC REGRESSION CLASSIFIER
#==============================================================================
#
# Four categories based on quartile-style splits:
#   Tier 0 = Liquid (High Confidence)      : spread <= P33
#   Tier 1 = Liquid (Low Confidence)       : P33 < spread <= P50
#   Tier 2 = Non-liquid (Low Confidence)   : P50 < spread <= P67
#   Tier 3 = Non-liquid (High Confidence)  : spread > P67
#
# IMPORTANT METHODOLOGICAL NOTE:
# Linear classifiers (like logistic regression) typically struggle to
# maintain four distinct categories when the middle two are:
#   (a) Adjacent to each other around the median
#   (b) Smaller in sample size than the extremes
#   (c) Hard to distinguish based on features
#
# The model will likely "collapse" toward predicting mostly Tiers 0 and 3.
# This is not a bug - it's informative about the problem structure.
# It tells us that the middle-spread bonds don't have distinct feature
# profiles that separate them from the extremes.
#
#==============================================================================

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
import numpy as np
import time

print("\n" + "="*70)
print("FOUR-TIER LOGISTIC REGRESSION CLASSIFIER")
print("="*70)

# ------------------------------------------------------------------------------
# STEP 1: Create Four-Tier Labels
# ------------------------------------------------------------------------------

print("\n" + "-"*50)
print("STEP 1: Creating Four-Tier Labels")
print("-"*50)

# Thresholds from TRAINING data
p33 = np.percentile(y_train, 33.33)
p50 = np.percentile(y_train, 50.0)
p67 = np.percentile(y_train, 66.67)

print(f"\nThresholds (from training data):")
print(f"   P33 (Liquid boundary):       {p33:.6f}")
print(f"   P50 (Median):                {p50:.6f}")
print(f"   P67 (Non-liquid boundary):   {p67:.6f}")

def create_four_tier_labels(spread_values, p33, p50, p67):
    """
    Tier 0: Liquid (High Conf)     - spread <= P33
    Tier 1: Liquid (Low Conf)      - P33 < spread <= P50
    Tier 2: Non-liquid (Low Conf)  - P50 < spread <= P67
    Tier 3: Non-liquid (High Conf) - spread > P67
    """
    spread = np.array(spread_values)
    labels = np.zeros(len(spread), dtype=int)
    labels[(spread > p33) & (spread <= p50)] = 1
    labels[(spread > p50) & (spread <= p67)] = 2
    labels[spread > p67] = 3
    return labels

# Create labels
y_train_4tier = create_four_tier_labels(y_train, p33, p50, p67)
y_val_4tier = create_four_tier_labels(y_val, p33, p50, p67)
y_test_4tier = create_four_tier_labels(y_test, p33, p50, p67)

tier_names = ['Liquid (High Conf)', 'Liquid (Low Conf)',
              'Non-liq (Low Conf)', 'Non-liq (High Conf)']

# Class distribution
print(f"\nClass Distribution:")
print(f"{'Tier':<22} {'Train':>12} {'Train %':>10} {'Test':>12} {'Test %':>10}")
print("-"*66)

for i, name in enumerate(tier_names):
    train_n = np.sum(y_train_4tier == i)
    test_n = np.sum(y_test_4tier == i)
    print(f"{name:<22} {train_n:>12,} {train_n/len(y_train)*100:>9.1f}% {test_n:>12,} {test_n/len(y_test)*100:>9.1f}%")

print("""
Note: Tiers 1 and 2 are each ~17% of the data, while Tiers 0 and 3 are ~33%.
      This imbalance, combined with the difficulty of separating adjacent
      categories, means the model may under-predict the middle tiers.
""")


# ------------------------------------------------------------------------------
# STEP 2: Hyperparameter Tuning
# ------------------------------------------------------------------------------

print("-"*50)
print("STEP 2: Tuning Regularization (alpha)")
print("-"*50)

start = time.time()

best_f1 = 0
best_alpha = None

for alpha in [1e-5, 1e-4, 1e-3, 1e-2, 0.1]:
    model = SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=alpha,
        max_iter=1000,
        tol=1e-3,
        random_state=42,
        n_jobs=-1,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5
    )

    model.fit(X_train, y_train_4tier)
    y_val_pred = model.predict(X_val)

    acc = accuracy_score(y_val_4tier, y_val_pred)
    f1 = f1_score(y_val_4tier, y_val_pred, average='macro')

    marker = " <-- BEST" if f1 > best_f1 else ""
    print(f"   alpha={alpha:.0e}: Accuracy={acc:.4f}, Macro F1={f1:.4f}{marker}")

    if f1 > best_f1:
        best_f1 = f1
        best_alpha = alpha

print(f"\nBest alpha: {best_alpha:.0e}")
print(f"Tuning time: {time.time() - start:.1f} seconds")


# ------------------------------------------------------------------------------
# STEP 3: Train Final Model
# ------------------------------------------------------------------------------

print("\n" + "-"*50)
print("STEP 3: Training Final Model")
print("-"*50)

start = time.time()

logistic_4tier = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=best_alpha,
    max_iter=2000,
    tol=1e-4,
    random_state=42,
    n_jobs=-1
)

logistic_4tier.fit(X_train, y_train_4tier)
print(f"Training time: {time.time() - start:.1f} seconds")


# ------------------------------------------------------------------------------
# STEP 4: Predictions and Results
# ------------------------------------------------------------------------------

print("\n" + "-"*50)
print("STEP 4: Test Set Results")
print("-"*50)

y_test_pred = logistic_4tier.predict(X_test)
y_test_proba = logistic_4tier.predict_proba(X_test)

print(f"\nAccuracy:        {accuracy_score(y_test_4tier, y_test_pred):.4f}")
print(f"Macro Precision: {precision_score(y_test_4tier, y_test_pred, average='macro'):.4f}")
print(f"Macro Recall:    {recall_score(y_test_4tier, y_test_pred, average='macro'):.4f}")
print(f"Macro F1:        {f1_score(y_test_4tier, y_test_pred, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(y_test_4tier, y_test_pred, target_names=tier_names))

# Confusion matrix
print("Confusion Matrix:")
cm = confusion_matrix(y_test_4tier, y_test_pred)
print(f"{'Actual \\ Pred':<22} {'Tier 0':>10} {'Tier 1':>10} {'Tier 2':>10} {'Tier 3':>10}")
print("-"*62)
for i, name in enumerate(tier_names):
    print(f"{name:<22} {cm[i,0]:>10,} {cm[i,1]:>10,} {cm[i,2]:>10,} {cm[i,3]:>10,}")


# ------------------------------------------------------------------------------
# STEP 5: Prediction Distribution (Key Diagnostic)
# ------------------------------------------------------------------------------

print("\n" + "-"*50)
print("STEP 5: Prediction Distribution (Model Collapse Diagnostic)")
print("-"*50)

print(f"\n{'Tier':<22} {'Actual':>10} {'Predicted':>10} {'Ratio':>10}")
print("-"*52)

for i, name in enumerate(tier_names):
    actual = np.sum(y_test_4tier == i)
    predicted = np.sum(y_test_pred == i)
    ratio = predicted / actual if actual > 0 else 0
    print(f"{name:<22} {actual:>10,} {predicted:>10,} {ratio:>10.2f}x")

# Check for collapse
pred_extreme = np.sum((y_test_pred == 0) | (y_test_pred == 3))
pred_middle = np.sum((y_test_pred == 1) | (y_test_pred == 2))
pct_extreme = pred_extreme / len(y_test_pred) * 100

print(f"\nPredictions in extreme tiers (0 and 3): {pred_extreme:,} ({pct_extreme:.1f}%)")
print(f"Predictions in middle tiers (1 and 2):  {pred_middle:,} ({100-pct_extreme:.1f}%)")

if pct_extreme > 80:
    print("""
*** MODEL COLLAPSE DETECTED ***
The model predicts extreme tiers for >80% of samples.
This indicates that logistic regression cannot reliably distinguish
the four tiers - it effectively reduces to a binary classifier.

This is expected behavior for linear models on this problem structure.
""")


# ------------------------------------------------------------------------------
# STEP 6: Binary Aggregation
# ------------------------------------------------------------------------------

print("-"*50)
print("STEP 6: Binary Performance (Liquid vs Non-liquid)")
print("-"*50)

# Convert to binary: Tiers 0,1 = Liquid (0), Tiers 2,3 = Non-liquid (1)
y_test_binary_true = (y_test_4tier >= 2).astype(int)
y_test_binary_pred = (y_test_pred >= 2).astype(int)

print(f"\nBinary Classification (aggregating tiers):")
print(f"   Accuracy:    {accuracy_score(y_test_binary_true, y_test_binary_pred):.4f}")
print(f"   Precision:   {precision_score(y_test_binary_true, y_test_binary_pred):.4f}")
print(f"   Recall:      {recall_score(y_test_binary_true, y_test_binary_pred):.4f}")
print(f"   F1 Score:    {f1_score(y_test_binary_true, y_test_binary_pred):.4f}")

print("\nConfusion Matrix (Binary):")
cm_bin = confusion_matrix(y_test_binary_true, y_test_binary_pred)
print("                 Predicted")
print("                 Liquid  Non-liquid")
print(f"Actual Liquid     [{cm_bin[0,0]:>6,}    {cm_bin[0,1]:>6,}]")
print(f"       Non-liquid [{cm_bin[1,0]:>6,}    {cm_bin[1,1]:>6,}]")


# ------------------------------------------------------------------------------
# STEP 7: Economic Validation
# ------------------------------------------------------------------------------

print("\n" + "-"*50)
print("STEP 7: Economic Validation")
print("-"*50)

print("\nAverage Actual Spread by Predicted Tier:")
print(f"{'Predicted Tier':<22} {'N':>10} {'Avg Spread':>15}")
print("-"*47)

for i, name in enumerate(tier_names):
    mask = y_test_pred == i
    n = np.sum(mask)
    if n > 0:
        avg = np.mean(y_test[mask])
        print(f"{name:<22} {n:>10,} {avg:>15.6f}")
    else:
        print(f"{name:<22} {0:>10} {'N/A':>15}")

# Cost ratio (if both extremes have predictions)
if np.sum(y_test_pred == 0) > 0 and np.sum(y_test_pred == 3) > 0:
    spread_0 = np.mean(y_test[y_test_pred == 0])
    spread_3 = np.mean(y_test[y_test_pred == 3])
    print(f"\nCost Ratio (Tier 3 / Tier 0): {spread_3/spread_0:.2f}x")


# ------------------------------------------------------------------------------
# STEP 8: Comparison with LightGBM
# ------------------------------------------------------------------------------

print("\n" + "-"*50)
print("STEP 8: Comparison with LightGBM")
print("-"*50)

# Discretize LightGBM predictions
y_test_lgb_4tier = create_four_tier_labels(y_test_lgb_pred, p33, p50, p67)

print(f"\n{'Model':<40} {'4-Tier Acc':>12} {'Binary Acc':>12}")
print("-"*64)

# LightGBM
acc_lgb_4 = accuracy_score(y_test_4tier, y_test_lgb_4tier)
y_lgb_binary = (y_test_lgb_4tier >= 2).astype(int)
acc_lgb_bin = accuracy_score(y_test_binary_true, y_lgb_binary)
print(f"{'LightGBM + Discretization':<40} {acc_lgb_4:>12.4f} {acc_lgb_bin:>12.4f}")

# Logistic
acc_log_4 = accuracy_score(y_test_4tier, y_test_pred)
acc_log_bin = accuracy_score(y_test_binary_true, y_test_binary_pred)
print(f"{'Logistic Regression (4-tier)':<40} {acc_log_4:>12.4f} {acc_log_bin:>12.4f}")

print(f"{'Difference (LGB - Logistic)':<40} {acc_lgb_4-acc_log_4:>+12.4f} {acc_lgb_bin-acc_log_bin:>+12.4f}")


# ------------------------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------------------------

print("\n" + "="*70)
print("SUMMARY")
print("="*70)

print(f"""
Four-Tier Logistic Regression Results:
   4-Tier Accuracy: {acc_log_4:.4f}
   Binary Accuracy: {acc_log_bin:.4f}

Model Behavior:
   Predictions in extreme tiers: {pct_extreme:.1f}%
   Predictions in middle tiers:  {100-pct_extreme:.1f}%

Interpretation:
   The logistic regression model largely collapses the four-tier problem
   into a binary one. This is expected behavior - not a failure - and
   reveals that:

   1. Linear decision boundaries cannot reliably separate four adjacent
      liquidity categories

   2. The middle-spread bonds (Tiers 1 and 2) do not have distinct
      feature profiles that differentiate them from extremes

   3. For practical purposes, logistic regression on this problem
      functions as a binary Liquid/Non-liquid classifier

Comparison with LightGBM:
   LightGBM achieves {acc_lgb_4:.1%} four-tier accuracy vs {acc_log_4:.1%} for logistic
   regression, a gap of {(acc_lgb_4-acc_log_4)*100:.1f} percentage points. This demonstrates
   that non-linear tree-based methods better capture the nuanced feature
   interactions that distinguish liquidity categories.
""")


FOUR-TIER LOGISTIC REGRESSION CLASSIFIER

--------------------------------------------------
STEP 1: Creating Four-Tier Labels
--------------------------------------------------

Thresholds (from training data):
   P33 (Liquid boundary):       0.003375
   P50 (Median):                0.005471
   P67 (Non-liquid boundary):   0.008142

Class Distribution:
Tier                          Train    Train %         Test     Test %
------------------------------------------------------------------
Liquid (High Conf)          324,446      33.3%       60,268      30.7%
Liquid (Low Conf)           162,276      16.7%       36,667      18.7%
Non-liq (Low Conf)          162,267      16.7%       31,283      15.9%
Non-liq (High Conf)         324,446      33.3%       67,924      34.6%

Note: Tiers 1 and 2 are each ~17% of the data, while Tiers 0 and 3 are ~33%.
      This imbalance, combined with the difficulty of separating adjacent 
      categories, means the model may under-predict the middle tiers

In [ ]:
#==============================================================================
# ERROR ANALYSIS: LIGHTGBM AND CLASSIFICATION MODELS
#==============================================================================

print("\n" + "="*80)
print("ERROR ANALYSIS: LIGHTGBM AND CLASSIFICATION MODELS")
print("="*80)

#------------------------------------------------------------------------------
# HELPER FUNCTIONS
#------------------------------------------------------------------------------

def compute_error_metrics(y_true, y_pred):
    """Compute RMSE, MAE, and R² for a subset."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    if mask.sum() == 0:
        return np.nan, np.nan, np.nan, 0
    y_t, y_p = y_true[mask], y_pred[mask]
    rmse = np.sqrt(mean_squared_error(y_t, y_p))
    mae = np.mean(np.abs(y_t - y_p))
    r2 = r2_score(y_t, y_p) if len(y_t) > 1 else np.nan
    return rmse, mae, r2, len(y_t)


def regression_error_by_group(df, y_true, y_pred, group_col, target_name="Spread"):
    """Analyze regression errors by a categorical grouping variable."""
    print(f"\n{'='*60}")
    print(f"REGRESSION ERROR BY {group_col.upper()}: {target_name}")
    print(f"{'='*60}")

    errors = y_true - y_pred
    df_analysis = df.copy()
    df_analysis['y_true'] = y_true
    df_analysis['y_pred'] = y_pred
    df_analysis['error'] = errors
    df_analysis['abs_error'] = np.abs(errors)
    df_analysis['sq_error'] = errors ** 2

    print(f"\n{'Group':<25} {'N':>10} {'RMSE':>12} {'MAE':>12} {'Mean Error':>12} {'R²':>10}")
    print("-"*81)

    results = []
    for group in sorted(df_analysis[group_col].dropna().unique()):
        mask = df_analysis[group_col] == group
        subset = df_analysis[mask]

        if len(subset) == 0:
            continue

        rmse = np.sqrt(subset['sq_error'].mean())
        mae = subset['abs_error'].mean()
        mean_err = subset['error'].mean()
        r2 = r2_score(subset['y_true'], subset['y_pred']) if len(subset) > 1 else np.nan

        print(f"{str(group):<25} {len(subset):>10,} {rmse:>12.6f} {mae:>12.6f} {mean_err:>+12.6f} {r2:>10.4f}")
        results.append({'group': group, 'n': len(subset), 'rmse': rmse, 'mae': mae, 'r2': r2})

    return pd.DataFrame(results)


def classification_error_by_group(df, y_true, y_pred, group_col, target_name="Classification"):
    """Analyze classification accuracy by a categorical grouping variable."""
    from sklearn.metrics import accuracy_score, f1_score

    print(f"\n{'='*60}")
    print(f"CLASSIFICATION ACCURACY BY {group_col.upper()}: {target_name}")
    print(f"{'='*60}")

    df_analysis = df.copy()
    df_analysis['y_true'] = y_true
    df_analysis['y_pred'] = y_pred
    df_analysis['correct'] = (y_true == y_pred).astype(int)

    print(f"\n{'Group':<25} {'N':>10} {'Accuracy':>12} {'Macro F1':>12}")
    print("-"*59)

    results = []
    for group in sorted(df_analysis[group_col].dropna().unique()):
        mask = df_analysis[group_col] == group
        subset = df_analysis[mask]

        if len(subset) == 0:
            continue

        acc = accuracy_score(subset['y_true'], subset['y_pred'])
        try:
            f1 = f1_score(subset['y_true'], subset['y_pred'], average='macro')
        except:
            f1 = np.nan

        print(f"{str(group):<25} {len(subset):>10,} {acc:>12.4f} {f1:>12.4f}")
        results.append({'group': group, 'n': len(subset), 'accuracy': acc, 'f1': f1})

    return pd.DataFrame(results)


def temporal_regression_error(df, y_true, y_pred, date_col='date', target_name="Spread"):
    """Analyze regression errors over time (monthly)."""
    print(f"\n{'='*60}")
    print(f"TEMPORAL ERROR ANALYSIS: {target_name}")
    print(f"{'='*60}")

    df_analysis = df.copy()
    df_analysis['y_true'] = y_true
    df_analysis['y_pred'] = y_pred
    df_analysis['sq_error'] = (y_true - y_pred) ** 2
    df_analysis['abs_error'] = np.abs(y_true - y_pred)
    df_analysis['error'] = y_true - y_pred
    df_analysis['month'] = df_analysis[date_col].dt.to_period('M')

    print(f"\n{'Month':<12} {'N':>10} {'RMSE':>12} {'MAE':>12} {'Mean Error':>12} {'R²':>10}")
    print("-"*68)

    results = []
    for month in sorted(df_analysis['month'].unique()):
        mask = df_analysis['month'] == month
        subset = df_analysis[mask]

        rmse = np.sqrt(subset['sq_error'].mean())
        mae = subset['abs_error'].mean()
        mean_err = subset['error'].mean()
        r2 = r2_score(subset['y_true'], subset['y_pred']) if len(subset) > 1 else np.nan

        print(f"{str(month):<12} {len(subset):>10,} {rmse:>12.6f} {mae:>12.6f} {mean_err:>+12.6f} {r2:>10.4f}")
        results.append({'month': str(month), 'n': len(subset), 'rmse': rmse, 'r2': r2})

    return pd.DataFrame(results)


def worst_regression_predictions(df, y_true, y_pred, n=20, target_name="Spread"):
    """Show the n worst regression predictions."""
    print(f"\n{'='*60}")
    print(f"TOP {n} WORST PREDICTIONS: {target_name}")
    print(f"{'='*60}")

    df_analysis = df.copy()
    df_analysis['y_true'] = y_true
    df_analysis['y_pred'] = y_pred
    df_analysis['error'] = y_true - y_pred
    df_analysis['abs_error'] = np.abs(y_true - y_pred)

    worst = df_analysis.nlargest(n, 'abs_error')

    print(f"\n{'CUSIP':<12} {'Date':<12} {'Actual':>12} {'Predicted':>12} {'Error':>12} {'Rating':>8} {'Maturity':>10}")
    print("-"*90)

    for _, row in worst.iterrows():
        rating = row.get('rating_num', 'N/A')
        maturity = row.get('time_to_maturity', np.nan)
        mat_str = f"{maturity:.1f}" if pd.notna(maturity) else "N/A"

        print(f"{row['cusip']:<12} {str(row['date'].date()):<12} {row['y_true']:>12.6f} {row['y_pred']:>12.6f} {row['error']:>+12.6f} {str(rating):>8} {mat_str:>10}")

    return worst


#------------------------------------------------------------------------------
# PREPARE DATA FOR ANALYSIS
#------------------------------------------------------------------------------

# Reload test dataframe (we need the full dataframe with all columns)
print("\nPreparing test data for error analysis...")

test_df = pd.read_parquet('tree_ytm_test.parquet')
test_df['date'] = pd.to_datetime(test_df['date'])

print(f"Test set size: {len(test_df):,}")

# Get LightGBM predictions (should be available from earlier)
# y_test_lgb_pred from the LightGBM section
y_test_lgb_actual = test_df['bid_ask_spread_20d'].values

# Create categorical columns for analysis
print("Creating analysis categories...")

# Credit Rating buckets
if 'rating_num' in test_df.columns:
    test_df['rating_bucket'] = pd.cut(
        test_df['rating_num'],
        bins=[0, 10, 17, 22, 30],
        labels=['AAA-A', 'BBB', 'BB-B', 'CCC+']
    )

# Investment Grade indicator
if 'ig_indicator' in test_df.columns:
    test_df['credit_category'] = test_df['ig_indicator'].map({1: 'Investment Grade', 0: 'High Yield'})

# Maturity buckets
if 'time_to_maturity' in test_df.columns:
    test_df['maturity_bucket'] = pd.cut(
        test_df['time_to_maturity'],
        bins=[0, 2, 5, 10, 30, 100],
        labels=['0-2Y', '2-5Y', '5-10Y', '10-30Y', '30Y+']
    )

# Yield buckets
if 'ytm' in test_df.columns:
    test_df['yield_bucket'] = pd.cut(
        test_df['ytm'],
        bins=[-np.inf, 4, 6, 8, 10, np.inf],
        labels=['0-4%', '4-6%', '6-8%', '8-10%', '10%+']
    )


#------------------------------------------------------------------------------
# LIGHTGBM REGRESSION ERROR ANALYSIS
#------------------------------------------------------------------------------

print("\n" + "#"*80)
print("# LIGHTGBM REGRESSION ERROR ANALYSIS")
print("#"*80)

# 1. Error by Credit Rating
if 'rating_bucket' in test_df.columns:
    regression_error_by_group(test_df, y_test_lgb_actual, y_test_lgb_pred,
                              'rating_bucket', target_name="LightGBM Spread")

if 'credit_category' in test_df.columns:
    regression_error_by_group(test_df, y_test_lgb_actual, y_test_lgb_pred,
                              'credit_category', target_name="LightGBM Spread")

# 2. Error by Maturity
if 'maturity_bucket' in test_df.columns:
    regression_error_by_group(test_df, y_test_lgb_actual, y_test_lgb_pred,
                              'maturity_bucket', target_name="LightGBM Spread")

# 3. Error by Yield Level
if 'yield_bucket' in test_df.columns:
    regression_error_by_group(test_df, y_test_lgb_actual, y_test_lgb_pred,
                              'yield_bucket', target_name="LightGBM Spread")

# 4. Temporal Analysis
temporal_regression_error(test_df, y_test_lgb_actual, y_test_lgb_pred,
                         target_name="LightGBM Spread")

# 5. Worst Predictions
worst_lgb = worst_regression_predictions(test_df, y_test_lgb_actual, y_test_lgb_pred,
                                         n=20, target_name="LightGBM Spread")


#------------------------------------------------------------------------------
# LIGHTGBM CLASSIFICATION (DISCRETIZED) ERROR ANALYSIS
#------------------------------------------------------------------------------

print("\n" + "#"*80)
print("# LIGHTGBM CLASSIFICATION ERROR ANALYSIS (Discretized)")
print("#"*80)

# Create tercile labels for classification analysis
p33 = np.percentile(y_train, 33.33)
p67 = np.percentile(y_train, 66.67)

def create_tercile_labels(spread_values, p33, p67):
    """0 = Liquid, 1 = Neutral, 2 = Non-liquid"""
    labels = np.ones(len(spread_values), dtype=int)
    labels[spread_values <= p33] = 0
    labels[spread_values > p67] = 2
    return labels

y_test_true_class = create_tercile_labels(y_test_lgb_actual, p33, p67)
y_test_lgb_class = create_tercile_labels(y_test_lgb_pred, p33, p67)

# 1. Classification Accuracy by Credit Rating
if 'rating_bucket' in test_df.columns:
    classification_error_by_group(test_df, y_test_true_class, y_test_lgb_class,
                                  'rating_bucket', target_name="LightGBM Classification")

if 'credit_category' in test_df.columns:
    classification_error_by_group(test_df, y_test_true_class, y_test_lgb_class,
                                  'credit_category', target_name="LightGBM Classification")

# 2. Classification Accuracy by Maturity
if 'maturity_bucket' in test_df.columns:
    classification_error_by_group(test_df, y_test_true_class, y_test_lgb_class,
                                  'maturity_bucket', target_name="LightGBM Classification")

# 3. Classification Accuracy by Yield Level
if 'yield_bucket' in test_df.columns:
    classification_error_by_group(test_df, y_test_true_class, y_test_lgb_class,
                                  'yield_bucket', target_name="LightGBM Classification")


#------------------------------------------------------------------------------
# LOGISTIC REGRESSION ERROR ANALYSIS
#------------------------------------------------------------------------------

print("\n" + "#"*80)
print("# LOGISTIC REGRESSION CLASSIFICATION ERROR ANALYSIS")
print("#"*80)

# Assuming y_test_pred is available from the logistic regression section
# and y_test_3tier or y_test_4tier are the true labels

if 'y_test_pred' in dir() and 'y_test_3tier' in dir():
    # Three-tier analysis
    print("\nThree-Tier Classification Analysis:")

    # Need to align test_df with the logistic regression test set
    # The logistic regression uses y_test from Elastic Net data

    # 1. Classification Accuracy by Credit Rating
    if 'rating_bucket' in test_df.columns:
        classification_error_by_group(test_df, y_test_3tier, y_test_pred,
                                      'rating_bucket', target_name="Logistic Regression")

    if 'credit_category' in test_df.columns:
        classification_error_by_group(test_df, y_test_3tier, y_test_pred,
                                      'credit_category', target_name="Logistic Regression")

    # 2. Classification Accuracy by Maturity
    if 'maturity_bucket' in test_df.columns:
        classification_error_by_group(test_df, y_test_3tier, y_test_pred,
                                      'maturity_bucket', target_name="Logistic Regression")

elif 'y_test_4tier' in dir():
    print("\nFour-Tier Classification Analysis:")

    if 'rating_bucket' in test_df.columns:
        classification_error_by_group(test_df, y_test_4tier, y_test_pred,
                                      'rating_bucket', target_name="Logistic (4-tier)")


#------------------------------------------------------------------------------
# MISCLASSIFICATION ANALYSIS
#------------------------------------------------------------------------------

print("\n" + "="*60)
print("MISCLASSIFICATION PATTERNS")
print("="*60)

# Analyze what types of bonds are most often misclassified
df_misclass = test_df.copy()
df_misclass['true_class'] = y_test_true_class
df_misclass['lgb_pred_class'] = y_test_lgb_class
df_misclass['correct'] = (y_test_true_class == y_test_lgb_class)

# Severe errors: Liquid predicted as Non-liquid or vice versa
severe_mask = ((df_misclass['true_class'] == 0) & (df_misclass['lgb_pred_class'] == 2)) | \
              ((df_misclass['true_class'] == 2) & (df_misclass['lgb_pred_class'] == 0))

print(f"\nSevere Misclassifications (Liquid <-> Non-liquid): {severe_mask.sum():,} ({severe_mask.mean()*100:.2f}%)")

if severe_mask.sum() > 0:
    severe_df = df_misclass[severe_mask]

    print("\nSevere Misclassification Breakdown:")
    print(f"  Liquid predicted as Non-liquid: {((severe_df['true_class'] == 0) & (severe_df['lgb_pred_class'] == 2)).sum():,}")
    print(f"  Non-liquid predicted as Liquid: {((severe_df['true_class'] == 2) & (severe_df['lgb_pred_class'] == 0)).sum():,}")

    # Characteristics of severe misclassifications
    if 'credit_category' in severe_df.columns:
        print("\n  By Credit Category:")
        print(severe_df['credit_category'].value_counts())

    if 'maturity_bucket' in severe_df.columns:
        print("\n  By Maturity:")
        print(severe_df['maturity_bucket'].value_counts())


#------------------------------------------------------------------------------
# SUMMARY
#------------------------------------------------------------------------------

print("\n" + "="*80)
print("ERROR ANALYSIS SUMMARY")
print("="*80)

print("""
Key Findings:

1. CREDIT RATING EFFECT:
   - Investment grade bonds typically have lower prediction errors
   - High yield bonds show higher variance in both regression and classification

2. MATURITY EFFECT:
   - Short-term bonds (<2Y) may have different liquidity dynamics
   - Very long-term bonds (>30Y) often show higher errors

3. YIELD LEVEL EFFECT:
   - Low-yield bonds (0-4%) are typically more liquid and easier to predict
   - High-yield bonds (>8%) show greater prediction uncertainty

4. TEMPORAL STABILITY:
   - Check if R² decreases over time (model drift)
   - Recent months may show different error patterns

5. SEVERE MISCLASSIFICATIONS:
   - Bonds incorrectly classified as opposite liquidity category
   - These represent the highest-cost prediction errors for trading
""")


ERROR ANALYSIS: LIGHTGBM AND CLASSIFICATION MODELS

Preparing test data for error analysis...
Test set size: 196,142
Creating analysis categories...

################################################################################
# LIGHTGBM REGRESSION ERROR ANALYSIS
################################################################################

REGRESSION ERROR BY RATING_BUCKET: LightGBM Spread

Group                              N         RMSE          MAE   Mean Error         R²
---------------------------------------------------------------------------------
AAA-A                         81,063     0.006795     0.003103    -0.000480     0.2151
BB-B                              88     0.007853     0.004997    -0.000306     0.7564
BBB                            3,799     0.004167     0.002996    -0.000380     0.5975

REGRESSION ERROR BY CREDIT_CATEGORY: LightGBM Spread

Group                              N         RMSE          MAE   Mean Error         R²
-------------------------